This notebook uploads page-level transcriptions from `04.CoMMa/00.data/02.csvs_arkindex/00.exact_page_nmb_match` and `04.CoMMa/00.data/02.csvs_arkindex/03.last_img_absent_in_CoMMA` to Arkindex.

The CSV filenames are Arkindex manuscript IDs. Each CSV row already contains the target page id in `arkindex_page_id` and the text to publish in `text`.

## 1. Environment

In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import os
import time
import uuid

import pandas as pd
import requests

In [21]:
os.chdir("C:\\Users\\yatsyk\\Documents\\NextCloud_Synchro\\")

## 2. Configuration

The upload cell will only write to Arkindex when `DRY_RUN = False`.

In [ ]:
ARKINDEX_BASE_URL = "https://demo.arkindex.org"
TOKEN_PATH = Path("TEKLIA/notebooks/token_API.txt")

CSV_DIRS = [
    Path("passim_BoH/04.CoMMa/00.data/03.csvs_arkindex/EXACT_MATCH")
]
REPORT_DIR = Path("passim_BoH/04.CoMMa/02.metadata/arkindex_transcription_upload")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# This file must contain the User WorkerRun ID created for the CoMMA transcription import.
# The same ID is used twice below: first to find/delete old transcriptions, then to create new ones.
WORKER_ID_PATH = Path("TEKLIA/notebooks/worker_id_import_CoMMA_transcriptions.txt")
worker_id_import_CoMMA_transcriptions = WORKER_ID_PATH.read_text(encoding="utf-8").strip()
WORKER_RUN_ID = worker_id_import_CoMMA_transcriptions

DRY_RUN = False

DELETE_EXISTING_TRANSCRIPTIONS = False
BATCH_SIZE = 50
DELETE_BATCH_SIZE = 100
RESUME_UPLOAD_FROM_BATCH = 1
ORIENTATION = "horizontal-lr"
CONFIDENCE = 0.5
SKIP_EMPTY_TEXT = True
SLEEP_BETWEEN_BATCHES_SECONDS = 0.1

CSV_DIRS, REPORT_DIR, WORKER_ID_PATH


([WindowsPath('passim_BoH/04.CoMMa/00.data/03.csvs_arkindex/EXACT_MATCH')],
 WindowsPath('passim_BoH/04.CoMMa/02.metadata/arkindex_transcription_upload'),
 WindowsPath('TEKLIA/notebooks/worker_id_import_CoMMA_transcriptions.txt'))

In [23]:
ARKINDEX_TOKEN = TOKEN_PATH.read_text(encoding="utf-8").strip()

headers = {
    "Authorization": f"Token {ARKINDEX_TOKEN}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}


def arkindex_request(method: str, path: str, *, expected=(200,), **kwargs) -> requests.Response:
    url = f"{ARKINDEX_BASE_URL.rstrip('/')}{path}"
    response = requests.request(method, url, headers=headers, timeout=60, **kwargs)
    if response.status_code not in expected:
        print(f"{method} {url}")
        print(f"HTTP {response.status_code}")
        print(response.text[:4000])
        response.raise_for_status()
    return response


def validate_worker_run_id(worker_run_id: str) -> dict:
    """Check that the WorkerRun ID is a UUID and that Arkindex can retrieve it."""
    try:
        uuid.UUID(worker_run_id)
    except ValueError as exc:
        raise ValueError(
            "worker_id_import_CoMMA_transcriptions must contain a User WorkerRun UUID, "
            "not the API token and not the worker version ID."
        ) from exc

    response = arkindex_request("GET", f"/api/v1/process/workers/{worker_run_id}/", expected=(200,))
    return response.json()


# Quick authentication check.
corpus_response = arkindex_request("GET", "/api/v1/corpus/", expected=(200,))
print(f"Connected to {ARKINDEX_BASE_URL}: {len(corpus_response.json())} corpora visible")


Connected to https://demo.arkindex.org: 25 corpora visible


## 3. Load and validate CSV files

In [24]:
REQUIRED_COLUMNS = {"text", "arkindex_page_id"}


def read_transcription_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, dtype=str).fillna("")
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {sorted(missing)}")

    df = df.copy()
    df["manuscript_arkindex_id"] = csv_path.stem
    df["csv_path"] = str(csv_path)
    df["csv_row"] = range(2, len(df) + 2)  # CSV line number, including header.
    df["text_clean"] = df["text"].astype(str).str.strip()
    df["arkindex_page_id"] = df["arkindex_page_id"].astype(str).str.strip()
    return df

csv_paths = sorted(
    path
    for csv_dir in CSV_DIRS
    for path in csv_dir.glob("*.csv")
)

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found in {CSV_DIRS}")

all_rows = pd.concat([read_transcription_csv(path) for path in csv_paths], ignore_index=True)
print(f"Loaded {len(all_rows):,} rows from {len(csv_paths):,} CSV files")
all_rows.head()

Loaded 50,418 rows from 178 CSV files


,page_id,CoMMA_folio_name,CoMMA_transcribed_page_nmb,text,page_ordering,arkindex_page_id,arkindex_page_name,CoMMA_folio_name_vs_arkindex_page_name_status,manuscript_arkindex_id,csv_path,csv_row,text_clean
0,f0-np.xml,np,0,CE DOCUIAENT A ETE MICROEIINAE\nTEL OUIL AETE ...,1,0ca87a00-c3ad-462e-835c-b520d0653cea,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,2,CE DOCUIAENT A ETE MICROEIINAE\nTEL OUIL AETE ...
1,f1-np.xml,np,1,,2,30a45f0f-8de3-4879-b156-2aa3049d4331,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,3,
2,f2-np.xml,np,2,LATIN\nIOS,3,7c360765-3c01-411a-8b4f-1dcc1aecf98d,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,4,LATIN\nIOS
3,f3-np.xml,np,3,,4,32caf8d2-4abf-4ec7-a0b2-dc45e4920100,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,5,
4,f4-np.xml,np,4,,5,63088df2-ab1b-4ae4-948e-88a50b0c421a,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,6,


In [27]:
def is_valid_uuid(value: str) -> bool:
    try:
        uuid.UUID(str(value))
        return True
    except ValueError:
        return False


invalid_page_ids = all_rows[all_rows["arkindex_page_id"].eq("")].copy()
invalid_uuid_page_ids = all_rows[
    all_rows["arkindex_page_id"].ne("")
    & ~all_rows["arkindex_page_id"].map(is_valid_uuid)
].copy()
empty_text_rows = all_rows[all_rows["text_clean"].eq("")].copy()
duplicate_targets = all_rows[
    all_rows["arkindex_page_id"].ne("")
    & all_rows["text_clean"].ne("")
    & all_rows.duplicated("arkindex_page_id", keep=False)
].sort_values(["arkindex_page_id", "manuscript_arkindex_id", "csv_row"])

validation_summary = pd.DataFrame(
    [
        {"check": "csv_files", "count": len(csv_paths)},
        {"check": "all_rows", "count": len(all_rows)},
        {"check": "missing_arkindex_page_id", "count": len(invalid_page_ids)},
        {"check": "invalid_uuid_arkindex_page_id", "count": len(invalid_uuid_page_ids)},
        {"check": "empty_text_rows", "count": len(empty_text_rows)},
        {"check": "duplicate_nonempty_page_targets", "count": len(duplicate_targets)},
    ]
)

validation_summary

,check,count
0,csv_files,178
1,all_rows,50418
2,missing_arkindex_page_id,0
3,invalid_uuid_arkindex_page_id,0
4,empty_text_rows,2829
5,duplicate_nonempty_page_targets,0


In [28]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

invalid_page_ids.to_csv(REPORT_DIR / f"invalid_page_ids_{timestamp}.csv", index=False, encoding="utf-8-sig")
invalid_uuid_page_ids.to_csv(REPORT_DIR / f"invalid_uuid_page_ids_{timestamp}.csv", index=False, encoding="utf-8-sig")
empty_text_rows.to_csv(REPORT_DIR / f"empty_text_rows_{timestamp}.csv", index=False, encoding="utf-8-sig")
duplicate_targets.to_csv(REPORT_DIR / f"duplicate_targets_{timestamp}.csv", index=False, encoding="utf-8-sig")

if len(invalid_page_ids):
    raise ValueError(f"Found {len(invalid_page_ids)} rows without arkindex_page_id. See report directory.")

if len(invalid_uuid_page_ids):
    raise ValueError(
        f"Found {len(invalid_uuid_page_ids)} rows where arkindex_page_id is not a valid UUID. See report directory."
    )

if len(duplicate_targets):
    raise ValueError(f"Found {len(duplicate_targets)} duplicate non-empty page targets. See report directory.")

upload_df = all_rows[
    all_rows["arkindex_page_id"].ne("")
    & all_rows["arkindex_page_id"].map(is_valid_uuid)
].copy()
if SKIP_EMPTY_TEXT:
    upload_df = upload_df[upload_df["text_clean"].ne("")].copy()

upload_df = upload_df.reset_index(drop=True)
print(f"Prepared {len(upload_df):,} page transcriptions for upload")
upload_df[["manuscript_arkindex_id", "page_id", "CoMMA_folio_name", "page_ordering", "arkindex_page_id", "text_clean"]].head(10)

Prepared 47,589 page transcriptions for upload


,manuscript_arkindex_id,page_id,CoMMA_folio_name,page_ordering,arkindex_page_id,text_clean
0,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f0-np.xml,np,1,0ca87a00-c3ad-462e-835c-b520d0653cea,CE DOCUIAENT A ETE MICROEIINAE\nTEL OUIL AETE ...
1,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f2-np.xml,np,3,7c360765-3c01-411a-8b4f-1dcc1aecf98d,LATIN\nIOS
2,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f7-np.xml,np,8,2ad1d65b-e77b-4c65-903b-366abe068a4e,1408.
3,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f8-np.xml,np,9,f8ac2b1c-c565-493e-8e5e-d8e2a91cbf5a,Lamuera.\nxxxi iour\nsacirtũcisiõ\neit ẽ estiẽ...
4,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f9-np.xml,np,10,6f45115c-75be-480b-875c-4493a4600e2c,May a.\nxi. b Siacque\nxxxi. iour\nG aquillã\n...
5,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f10-np.xml,np,11,9e8b8ff7-3939-4c63-935b-1cc8474a9284,deptebre a\nxui I S̾ leus gile\noxo iours\n¶ S...
6,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f12-np.xml,np,13,218a03b5-8eaa-4da0-9c6f-1d3395425aaa,et uerbum erat apud deum\net deus erat uerbum ...
7,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f13-np.xml,np,14,fadf4870-f079-4695-b757-6aa13911d681,multiplica super nos misc\nricordiam tuam ut t...
8,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f14-np.xml,np,15,b4b20b8f-ffb7-4ad2-abea-bc7552b6d301,¶ Dissus est gabriel angelus\na deo in ciutate...
9,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,f15-np.xml,np,16,005d9a7c-c64c-4a5b-ad8c-1cad7b770498,tempus stelle que apairuit eis\nEt mitens eos ...


## 4. Build Arkindex payload

In [29]:
def build_transcription_item(row: pd.Series) -> dict:
    return {
        "element_id": row["arkindex_page_id"],
        "text": row["text_clean"],
        "confidence": CONFIDENCE,
        "orientation": ORIENTATION,
    }


payload_items = [build_transcription_item(row) for _, row in upload_df.iterrows()]
payload_preview = {
    "worker_run_id": WORKER_RUN_ID,
    "transcriptions": payload_items[:3],
}
payload_preview

{'worker_run_id': 'c59f8baa-5c7e-4316-9a77-e3b3f8de1201',
 'transcriptions': [{'element_id': '0ca87a00-c3ad-462e-835c-b520d0653cea',
   'text': 'CE DOCUIAENT A ETE MICROEIINAE\nTEL OUIL AETE RELIE',
   'confidence': 0.5,
   'orientation': 'horizontal-lr'},
  {'element_id': '7c360765-3c01-411a-8b4f-1dcc1aecf98d',
   'text': 'LATIN\nIOS',
   'confidence': 0.5,
   'orientation': 'horizontal-lr'},
  {'element_id': '2ad1d65b-e77b-4c65-903b-366abe068a4e',
   'text': '1408.',
   'confidence': 0.5,
   'orientation': 'horizontal-lr'}]}

## 5. Upload

In [30]:
def iter_batches(items: list[dict], size: int):
    for start in range(0, len(items), size):
        yield start, items[start : start + size]


def create_transcriptions_bulk(batch_items: list[dict]) -> dict:
    """Create a batch of page transcriptions attached to the configured WorkerRun."""
    payload = {
        "worker_run_id": WORKER_RUN_ID,
        "transcriptions": batch_items,
    }
    response = arkindex_request("POST", "/api/v1/transcription/bulk/", json=payload, expected=(201,))
    return response.json()


def list_page_transcriptions_for_worker_run(page_id: str, worker_run_id: str) -> list[dict]:
    """Return only transcriptions created by this WorkerRun on one Arkindex page element."""
    items = []
    path = f"/api/v1/element/{page_id}/transcriptions/"
    params = {"worker_run": worker_run_id, "page_size": 100}

    while path:
        response = arkindex_request("GET", path, params=params, expected=(200,))
        data = response.json()

        if isinstance(data, dict) and "results" in data:
            items.extend(data["results"])
            next_url = data.get("next")
            if next_url:
                path = next_url.replace(ARKINDEX_BASE_URL.rstrip("/"), "")
                params = None
            else:
                path = None
        elif isinstance(data, list):
            items.extend(data)
            path = None
        else:
            raise ValueError(f"Unexpected transcription list payload for page {page_id}: {data}")

    return items


def delete_transcription(transcription_id: str):
    """Delete one transcription by Arkindex transcription ID."""
    arkindex_request("DELETE", f"/api/v1/transcription/{transcription_id}/", expected=(204,))


def collect_existing_worker_transcriptions(page_ids: list[str], worker_run_id: str) -> list[dict]:
    """Collect old transcriptions from exactly this WorkerRun and exactly these page IDs."""
    existing = []
    for page_index, page_id in enumerate(page_ids, start=1):
        page_transcriptions = list_page_transcriptions_for_worker_run(page_id, worker_run_id)
        for transcription in page_transcriptions:
            existing.append(
                {
                    "page_id": page_id,
                    "transcription_id": transcription.get("id"),
                    "text": transcription.get("text", ""),
                    "worker_run_id": worker_run_id,
                }
            )

        if page_index % 250 == 0:
            print(f"Checked old transcriptions on {page_index:,}/{len(page_ids):,} target pages")

    return existing


def extract_created_transcriptions(response_payload) -> list[dict]:
    """Normalize Arkindex bulk-create responses to a list for logging."""
    if isinstance(response_payload, dict):
        created = response_payload.get("transcriptions", [])
        return created if isinstance(created, list) else []
    if isinstance(response_payload, list):
        return response_payload
    return []


upload_log = []
delete_log = []
preview_path = REPORT_DIR / f"payload_preview_{timestamp}.csv"
delete_log_path = REPORT_DIR / f"delete_log_{timestamp}.csv"
upload_log_path = REPORT_DIR / f"upload_log_{timestamp}.csv"

upload_df[["manuscript_arkindex_id", "csv_path", "csv_row", "arkindex_page_id", "text_clean"]].to_csv(
    preview_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Payload preview written to {preview_path}")
upload_df


Payload preview written to passim_BoH\04.CoMMa\02.metadata\arkindex_transcription_upload\payload_preview_20260605_110617.csv


,page_id,CoMMA_folio_name,CoMMA_transcribed_page_nmb,text,page_ordering,arkindex_page_id,arkindex_page_name,CoMMA_folio_name_vs_arkindex_page_name_status,manuscript_arkindex_id,csv_path,csv_row,text_clean
0,f0-np.xml,np,0,CE DOCUIAENT A ETE MICROEIINAE\nTEL OUIL AETE ...,1,0ca87a00-c3ad-462e-835c-b520d0653cea,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,2,CE DOCUIAENT A ETE MICROEIINAE\nTEL OUIL AETE ...
1,f2-np.xml,np,2,LATIN\nIOS,3,7c360765-3c01-411a-8b4f-1dcc1aecf98d,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,4,LATIN\nIOS
2,f7-np.xml,np,7,1408.,8,2ad1d65b-e77b-4c65-903b-366abe068a4e,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,9,1408.
3,f8-np.xml,np,8,Lamuera.\nxxxi iour\nsacirtũcisiõ\neit ẽ estiẽ...,9,f8ac2b1c-c565-493e-8e5e-d8e2a91cbf5a,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,10,Lamuera.\nxxxi iour\nsacirtũcisiõ\neit ẽ estiẽ...
4,f9-np.xml,np,9,May a.\nxi. b Siacque\nxxxi. iour\nG aquillã\n...,10,6f45115c-75be-480b-875c-4493a4600e2c,NP,NORMALIZED_MATCH_NP,003d5be0-3453-46e9-b904-4f6d4f1ee4b5,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,11,May a.\nxi. b Siacque\nxxxi. iour\nG aquillã\n...
...,...,...,...,...,...,...,...,...,...,...,...,...
47584,f261-128r.xml,128r,261,laudate eũ in psalterio et cytha\nraIaudate eũ...,262,3489f79d-6d08-4586-b715-7c3fda475b96,NP,MISMATCH,ff5abf5d-527f-4db5-9ae0-51222fe71a78,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,263,laudate eũ in psalterio et cytha\nraIaudate eũ...
47585,f262-128v.xml,128v,262,t erexit cornu salutis no\nbis in domo dauid p...,263,a78e9d2a-012f-4571-9e09-24ac0307fdc5,NP,MISMATCH,ff5abf5d-527f-4db5-9ae0-51222fe71a78,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,264,t erexit cornu salutis no\nbis in domo dauid p...
47586,f263-129r.xml,129r,263,cia coram ipso omnibus die\nbus nr̃is. ¶t tu p...,264,7228b205-d945-433e-942b-4a20422bfc7b,NP,MISMATCH,ff5abf5d-527f-4db5-9ae0-51222fe71a78,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,265,cia coram ipso omnibus die\nbus nr̃is. ¶t tu p...
47587,f264-129v.xml,129v,264,us fuerit uiuet et omnis qui\nuiuit credit in ...,265,6dc8d1e9-55d1-4b0c-9fc2-21c348d3e4f6,NP,MISMATCH,ff5abf5d-527f-4db5-9ae0-51222fe71a78,passim_BoH\04.CoMMa\00.data\03.csvs_arkindex\E...,266,us fuerit uiuet et omnis qui\nuiuit credit in ...


In [31]:
# Final import step: replace all transcriptions previously created by this WorkerRun,
# then upload the current CSV transcriptions using the very same WorkerRun ID.
#
# Important safety rule:
# We only list and delete transcriptions with the `worker_run` filter set to
# `worker_id_import_CoMMA_transcriptions`. This means manual transcriptions and
# transcriptions created by other workers are not targeted by this cell.

if not WORKER_RUN_ID:
    raise ValueError("WORKER_RUN_ID is empty. Check TEKLIA/notebooks/worker_id_import_CoMMA_transcriptions.txt")

full_upload_batch_count = (len(payload_items) + BATCH_SIZE - 1) // BATCH_SIZE
if RESUME_UPLOAD_FROM_BATCH < 1 or RESUME_UPLOAD_FROM_BATCH > full_upload_batch_count:
    raise ValueError(
        f"RESUME_UPLOAD_FROM_BATCH must be between 1 and {full_upload_batch_count}, got {RESUME_UPLOAD_FROM_BATCH}."
    )

if RESUME_UPLOAD_FROM_BATCH > 1 and DELETE_EXISTING_TRANSCRIPTIONS:
    raise ValueError(
        "DELETE_EXISTING_TRANSCRIPTIONS must stay False when resuming from a later batch. "
        "Otherwise already uploaded pages could be deleted."
    )

resume_start = (RESUME_UPLOAD_FROM_BATCH - 1) * BATCH_SIZE
run_upload_df = upload_df.iloc[resume_start:].copy()
run_payload_items = payload_items[resume_start:]

print("Step 1/6. WorkerRun selected for replacement")
print(f"WorkerRun ID: {WORKER_RUN_ID}")
print("This same ID will be used for deletion and for the new upload.")

print()
print("Step 2/6. Validating that the WorkerRun exists and is accessible")
worker_run = validate_worker_run_id(WORKER_RUN_ID)
print(worker_run.get("summary", "WorkerRun is accessible"))

print()
print("Step 3/6. Preparing the target page list")
target_page_ids = run_upload_df["arkindex_page_id"].drop_duplicates().tolist()
print(f"Full prepared rows: {len(upload_df):,}")
print(f"Resume from batch: {RESUME_UPLOAD_FROM_BATCH:,}/{full_upload_batch_count:,}")
print(f"Resume start row index: {resume_start:,}")
print(f"Rows to upload in this run: {len(run_upload_df):,}")
print(f"Distinct Arkindex page elements touched in this run: {len(target_page_ids):,}")
print(f"Local preview CSV: {preview_path}")

print()
print("Step 4/6. Looking for old transcriptions created by this WorkerRun")
if DELETE_EXISTING_TRANSCRIPTIONS:
    existing_transcriptions = collect_existing_worker_transcriptions(target_page_ids, WORKER_RUN_ID)
else:
    existing_transcriptions = []
    print("DELETE_EXISTING_TRANSCRIPTIONS=False, so the deletion search is skipped.")

print(
    f"Found {len(existing_transcriptions):,} old transcriptions created by WorkerRun {WORKER_RUN_ID} "
    f"on the target pages."
)

if existing_transcriptions:
    pd.DataFrame(existing_transcriptions).to_csv(
        REPORT_DIR / f"old_transcriptions_to_delete_{timestamp}.csv",
        index=False,
        encoding="utf-8-sig",
    )
    print(f"The deletion candidate list was saved in {REPORT_DIR}.")

if DRY_RUN:
    print()
    print("Step 5/6. DRY_RUN=True, so nothing is deleted and nothing is uploaded")
    print(
        f"Would delete {len(existing_transcriptions):,} old transcriptions from this WorkerRun, "
        f"then upload {len(run_payload_items):,} new transcriptions in this run."
    )
    print("Set DRY_RUN = False in the configuration cell when this preview looks correct.")
else:
    print()
    print("Step 5/6. Deleting old transcriptions from this WorkerRun")
    if DELETE_EXISTING_TRANSCRIPTIONS and existing_transcriptions:
        total_delete_batches = (len(existing_transcriptions) + DELETE_BATCH_SIZE - 1) // DELETE_BATCH_SIZE
        for delete_batch_index, (delete_start, delete_batch) in enumerate(
            iter_batches(existing_transcriptions, DELETE_BATCH_SIZE),
            start=1,
        ):
            for item in delete_batch:
                try:
                    delete_transcription(item["transcription_id"])
                    status = "OK"
                    error = ""
                except Exception as exc:
                    status = "ERROR"
                    error = repr(exc)

                delete_log.append(
                    {
                        "batch_index": delete_batch_index,
                        "page_id": item["page_id"],
                        "transcription_id": item["transcription_id"],
                        "worker_run_id": item["worker_run_id"],
                        "status": status,
                        "error": error,
                    }
                )

                if status == "ERROR":
                    pd.DataFrame(delete_log).to_csv(delete_log_path, index=False, encoding="utf-8-sig")
                    raise RuntimeError(
                        f"Deletion stopped on transcription {item['transcription_id']}. "
                        f"Partial log: {delete_log_path}"
                    )

            pd.DataFrame(delete_log).to_csv(delete_log_path, index=False, encoding="utf-8-sig")
            print(
                f"Delete batch {delete_batch_index:,}/{total_delete_batches:,}: "
                f"{len(delete_batch):,} transcriptions deleted"
            )
            time.sleep(SLEEP_BETWEEN_BATCHES_SECONDS)
    else:
        print("No old transcriptions to delete for this WorkerRun on the target pages.")

    print()
    print("Step 6/6. Uploading the new transcriptions with the same WorkerRun ID")
    print(f"Strict resume: starting at batch {RESUME_UPLOAD_FROM_BATCH:,}")
    for batch_index, (run_start, batch_items) in enumerate(
        iter_batches(run_payload_items, BATCH_SIZE),
        start=RESUME_UPLOAD_FROM_BATCH,
    ):
        start = resume_start + run_start
        source_slice = upload_df.iloc[start : start + len(batch_items)]
        try:
            result = create_transcriptions_bulk(batch_items)
            created = extract_created_transcriptions(result)
            status = "OK"
            error = ""
        except Exception as exc:
            created = []
            status = "ERROR"
            error = repr(exc)

        upload_log.append(
            {
                "batch_index": batch_index,
                "batch_start": start,
                "batch_size": len(batch_items),
                "status": status,
                "created_count": len(created),
                "worker_run_id": WORKER_RUN_ID,
                "first_page_id": source_slice.iloc[0]["arkindex_page_id"],
                "last_page_id": source_slice.iloc[-1]["arkindex_page_id"],
                "error": error,
            }
        )
        pd.DataFrame(upload_log).to_csv(upload_log_path, index=False, encoding="utf-8-sig")

        print(f"Upload batch {batch_index:,}/{full_upload_batch_count:,}: {status}, {len(created):,} created")
        if status == "ERROR":
            raise RuntimeError(f"Upload stopped on batch {batch_index}. Partial log: {upload_log_path}")

        time.sleep(SLEEP_BETWEEN_BATCHES_SECONDS)

    print()
    print("Replacement import complete")
    print(f"Deleted old transcriptions log: {delete_log_path}")
    print(f"New upload log: {upload_log_path}")


Step 1/6. WorkerRun selected for replacement
WorkerRun ID: c59f8baa-5c7e-4316-9a77-e3b3f8de1201
This same ID will be used for deletion and for the new upload.

Step 2/6. Validating that the WorkerRun exists and is accessible
import_CoMMA_transcriptions

Step 3/6. Preparing the target page list
Full prepared rows: 47,589
Resume from batch: 358/952
Resume start row index: 17,850
Rows to upload in this run: 29,739
Distinct Arkindex page elements touched in this run: 29,739
Local preview CSV: passim_BoH\04.CoMMa\02.metadata\arkindex_transcription_upload\payload_preview_20260605_110617.csv

Step 4/6. Looking for old transcriptions created by this WorkerRun
DELETE_EXISTING_TRANSCRIPTIONS=False, so the deletion search is skipped.
Found 0 old transcriptions created by WorkerRun c59f8baa-5c7e-4316-9a77-e3b3f8de1201 on the target pages.

Step 5/6. Deleting old transcriptions from this WorkerRun
No old transcriptions to delete for this WorkerRun on the target pages.

Step 6/6. Uploading the new t

# 6. Delete existing transcriptions

In [ ]:
# Separate cleanup cell: delete all page transcriptions in selected manuscripts
# that were created by the WorkerRun stored in WORKER_RUN_ID.
#
# Safety: keep MANUSCRIPT_DELETE_DRY_RUN=True for a preview. Set it to False only
# after checking the candidate CSV written by this cell.

MANUSCRIPT_IDS_CSV_PATH = Path("passim_BoH/04.CoMMa/02.metadata/196_arkindex_Comma_URL_filled.csv")
MANUSCRIPT_ID_COLUMN = "ID Arkindex"
MANUSCRIPT_DELETE_DRY_RUN = True


def list_manuscript_page_elements(manuscript_id: str) -> list[dict]:
    """Return all direct page children of one Arkindex manuscript element."""
    pages = []
    path = f"/api/v1/elements/{manuscript_id}/children/"
    params = {"type": "page", "order": "position", "page_size": 100}

    while path:
        response = arkindex_request("GET", path, params=params, expected=(200,))
        data = response.json()

        if isinstance(data, dict) and "results" in data:
            pages.extend(data["results"])
            next_url = data.get("next")
            if next_url:
                path = next_url.replace(ARKINDEX_BASE_URL.rstrip("/"), "")
                params = None
            else:
                path = None
        elif isinstance(data, list):
            pages.extend(data)
            path = None
        else:
            raise ValueError(f"Unexpected manuscript children payload for {manuscript_id}: {data}")

    return pages


if not WORKER_RUN_ID:
    raise ValueError("WORKER_RUN_ID is empty. Check WORKER_ID_PATH.")

manuscript_ids_df = pd.read_csv(MANUSCRIPT_IDS_CSV_PATH, dtype=str).fillna("")
if MANUSCRIPT_ID_COLUMN not in manuscript_ids_df.columns:
    raise ValueError(f"Missing column {MANUSCRIPT_ID_COLUMN!r} in {MANUSCRIPT_IDS_CSV_PATH}")

manuscript_ids_to_clear = (
    manuscript_ids_df[MANUSCRIPT_ID_COLUMN]
    .astype(str)
    .str.strip()
    .loc[lambda values: values.ne("")]
    .drop_duplicates()
    .tolist()
)
if not manuscript_ids_to_clear:
    raise ValueError(f"No Arkindex manuscript IDs found in {MANUSCRIPT_IDS_CSV_PATH}")

manuscript_delete_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
manuscript_candidate_path = REPORT_DIR / f"all_manuscripts_worker_run_transcriptions_to_delete_{manuscript_delete_timestamp}.csv"
manuscript_delete_log_path = REPORT_DIR / f"all_manuscripts_worker_run_delete_log_{manuscript_delete_timestamp}.csv"
manuscript_summary_path = REPORT_DIR / f"all_manuscripts_worker_run_delete_summary_{manuscript_delete_timestamp}.csv"

print("Step 1/4. Validating WorkerRun")
worker_run = validate_worker_run_id(WORKER_RUN_ID)
print(worker_run.get("summary", "WorkerRun is accessible"))
print(f"WorkerRun ID: {WORKER_RUN_ID}")

print()
print("Step 2/4. Loading manuscript IDs")
print(f"Source CSV: {MANUSCRIPT_IDS_CSV_PATH}")
print(f"Manuscripts to inspect: {len(manuscript_ids_to_clear):,}")

print()
print("Step 3/4. Looking for transcriptions from this WorkerRun on manuscript pages")
all_manuscript_transcriptions_to_delete = []
manuscript_cleanup_summary = []

for manuscript_index, manuscript_id in enumerate(manuscript_ids_to_clear, start=1):
    print()
    print(f"Manuscript {manuscript_index:,}/{len(manuscript_ids_to_clear):,}: {manuscript_id}")
    manuscript_pages = list_manuscript_page_elements(manuscript_id)
    manuscript_page_ids = [page["id"] for page in manuscript_pages]
    print(f"Pages found: {len(manuscript_page_ids):,}")

    manuscript_transcriptions_to_delete = collect_existing_worker_transcriptions(
        manuscript_page_ids,
        WORKER_RUN_ID,
    )
    for transcription in manuscript_transcriptions_to_delete:
        transcription["manuscript_id"] = manuscript_id
        transcription["manuscript_index"] = manuscript_index

    all_manuscript_transcriptions_to_delete.extend(manuscript_transcriptions_to_delete)
    manuscript_cleanup_summary.append(
        {
            "manuscript_index": manuscript_index,
            "manuscript_id": manuscript_id,
            "page_count": len(manuscript_page_ids),
            "transcriptions_to_delete": len(manuscript_transcriptions_to_delete),
        }
    )
    print(f"Transcriptions to delete in this manuscript: {len(manuscript_transcriptions_to_delete):,}")

pd.DataFrame(
    all_manuscript_transcriptions_to_delete,
    columns=["manuscript_index", "manuscript_id", "page_id", "transcription_id", "text", "worker_run_id"],
).to_csv(manuscript_candidate_path, index=False, encoding="utf-8-sig")
pd.DataFrame(manuscript_cleanup_summary).to_csv(manuscript_summary_path, index=False, encoding="utf-8-sig")
print()
print(f"Found {len(all_manuscript_transcriptions_to_delete):,} transcriptions to delete across all manuscripts.")
print(f"Candidate CSV: {manuscript_candidate_path}")
print(f"Summary CSV: {manuscript_summary_path}")

print()
print("Step 4/4. Deleting selected transcriptions")
if MANUSCRIPT_DELETE_DRY_RUN:
    print("MANUSCRIPT_DELETE_DRY_RUN=True, so nothing is deleted.")
    print("Set MANUSCRIPT_DELETE_DRY_RUN=False in this cell to perform the deletion.")
elif not all_manuscript_transcriptions_to_delete:
    print("No matching transcriptions to delete.")
else:
    manuscript_delete_log = []
    total_delete_batches = (len(all_manuscript_transcriptions_to_delete) + DELETE_BATCH_SIZE - 1) // DELETE_BATCH_SIZE
    for delete_batch_index, (delete_start, delete_batch) in enumerate(
        iter_batches(all_manuscript_transcriptions_to_delete, DELETE_BATCH_SIZE),
        start=1,
    ):
        for item in delete_batch:
            try:
                delete_transcription(item["transcription_id"])
                status = "OK"
                error = ""
            except Exception as exc:
                status = "ERROR"
                error = repr(exc)

            manuscript_delete_log.append(
                {
                    "batch_index": delete_batch_index,
                    "manuscript_index": item["manuscript_index"],
                    "manuscript_id": item["manuscript_id"],
                    "page_id": item["page_id"],
                    "transcription_id": item["transcription_id"],
                    "worker_run_id": item["worker_run_id"],
                    "status": status,
                    "error": error,
                }
            )

            if status == "ERROR":
                pd.DataFrame(manuscript_delete_log).to_csv(
                    manuscript_delete_log_path,
                    index=False,
                    encoding="utf-8-sig",
                )
                raise RuntimeError(
                    f"Deletion stopped on transcription {item['transcription_id']}. "
                    f"Partial log: {manuscript_delete_log_path}"
                )

        pd.DataFrame(manuscript_delete_log).to_csv(
            manuscript_delete_log_path,
            index=False,
            encoding="utf-8-sig",
        )
        print(
            f"Delete batch {delete_batch_index:,}/{total_delete_batches:,}: "
            f"{len(delete_batch):,} transcriptions deleted"
        )
        time.sleep(SLEEP_BETWEEN_BATCHES_SECONDS)

    print("Manuscript cleanup complete")
    print(f"Delete log: {manuscript_delete_log_path}")
